# 🏪 Retail360 — Star Schema ETL Pipeline

> **Data source:** [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) (`online_retail_II.xlsx`)
> **Output layer:** Gold-layer Snowflake Schema (dimensions + facts)
> **Output format:** CSV

---

## 📋 Table of Contents

1. [Source data description](#1-source-data-description)
2. [Configuration and imports](#2-configuration-and-imports)
3. [EXTRACT — Data loading](#3-extract)
4. [CLEAN — Data cleaning](#4-clean)
5. [DIMENSIONS — Dimension building](#5-dimensions)
6. [FACTS — Fact table building](#6-facts)
7. [SAVE — Saving results](#7-save)
8. [QUALITY CHECK — Quality check](#8-quality-check)
9. [Pipeline execution](#9-execution)

---

## 2. Configuration and imports



In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
from typing import Dict

### ⚙️ Main parameters

In [21]:
RAW_FILE   = "../data/raw/online_retail_II.xlsx"
OUTPUT_DIR = Path("star_schema")
SEED       = 42

### 🚫 Non-product codes (`EXCLUDE_CODES`)

Source data contains codes, that **do not** correspond to real products — to shipping fees, bank adjustments, test codes etc. We remove them from further processing:

| Category | Codes | Description |
|-----------|------|------|
| Shipping | `POST`, `POSTAGE` | Shipping fee |
| Adjustments | `DOT`, `D`, `M`, `MANUAL` | Manual corrections |
| Bank charges | `BANK CHARGES`, `BANKCHARGES` | Bank commissions |
| Marketplace | `AMAZONFEE` | Amazon commission |
| Other | `PADS`, `CRUK`, `S`, `B` | Miscellaneous technical codes |
| Adjustments | `ADJUST`, `ADJUST2` | Value adjustments |
| Tests | `TEST001`, `TEST002` | Test transactions |
| DCGS | `DCGS0003`–`DCGS0076` | Internal codes |

In [ ]:
EXCLUDE_CODES = {
    'POST', 'POSTAGE',
    'DOT', 'D',
    'M', 'MANUAL',
    'BANK CHARGES', 'BANKCHARGES',
    'PADS',
    'AMAZONFEE',
    'CRUK',
    'C2', 'C3',
    'S', 'B',
    'ADJUST', 'ADJUST2',
    'TEST001', 'TEST002',
    'DCGS0003', 'DCGS0004',
    'DCGS0066', 'DCGS0067', 'DCGS0068',
    'DCGS0069', 'DCGS0070', 'DCGS0071',
    'DCGS0072', 'DCGS0073', 'DCGS0076',
}



### 📝 Logging configuration

We set up logging with a readable format `HH:MM:SS │ LEVEL │ message`.

In [23]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s │ %(levelname)-7s │ %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ETL_REPORT')

---

## 3. EXTRACT — Raw data extraction <a id="3-extract"></a>

The Excel file contains **two sheets**:
- `Year 2009-2010` — transactions from December 2009 to December 2010
- `Year 2010-2011` — transactions from December 2010 to December 2011

Both sheets have the same structure. We load and combine them into a single DataFrame.

In [24]:
def extract() -> pd.DataFrame:
    """Load both sheets from Excel file and combine."""
    log.info("═══ EXTRACT ═══════════════════════════════════════")

    dfs = []
    for sheet in ['Year 2009-2010', 'Year 2010-2011']:
        df = pd.read_excel(RAW_FILE, sheet_name=sheet)
        log.info(f"  Sheet '{sheet}': {len(df):>9,} rows")
        dfs.append(df)

    raw = pd.concat(dfs, ignore_index=True)
    log.info(f"  TOTAL:                   {len(raw):>9,} rows")
    log.info(f"  Columns: {list(raw.columns)}")
    return raw

---

## 4. CLEAN — Data cleaning <a id="4-clean"></a>

The cleaning stage is crucial dla jakości wynikowego schematu gwiazdy. Wykonujemy następujące kroki:

### Cleaning steps sequence

| # | Step | Action | Impact on data |
|---|------|---------|---------------|
| 2a | Column standardization | Lowercase, underscores instead of spaces | 0 loss |
| 2b | Base types | Type conversion, filling descriptions | 0 loss |
| 2c | Guest handling | Guest flagging, `customer_id=0` | 0 loss (guests preserved!) |
| 2d | Returns flag | `is_return` = invoice zaczyna się od `C` | 0 loss |
| 2e | Non-product codes | Usunięcie kodów z `EXCLUDE_CODES` | ~several hundred rows |
| 2f | Price < 0 | Remove negative prices (not on returns) | minimal |
| 2g | Quantity == 0 | Remove zero quantities | minimal |
| 2h | Deduplication | Remove identical duplicates | minimal |
| 2i | Calculated columns | `line_total`, `date_key`, `hour` | 0 loss |

> ⚠️ **Ważne:** Goście (brak `Customer ID`) **nie są usuwani** — dostają `customer_id = 0` i flagę `is_registered = False`.

In [ ]:
def clean(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans source data.

    Key principles:
      → Guests (null customer_id) are NOT removed
      → Guests get customer_id = 0 (sentinel)
      → is_registered = flag on every row
      → Explicit EXCLUDE_CODES (no mask_short_alpha)
      → price < 0 (not ≤ 0, freebies stay)
      → Full loss audit trail
    """
    log.info("═══ CLEAN ═════════════════════════════════════════")
    df = raw.copy()
    audit = []
    audit.append(('RAW input', len(df)))

    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_'))

    rename = {
        'invoiceno':   'invoice',
        'invoicedate': 'invoice_date',
        'stockcode':   'stock_code',
        'unetcrice':   'price',
        'customerid':  'customer_id',
    }
    df.rename(columns={k: v for k, v in rename.items()
                       if k in df.columns}, inplace=True)

    log.info(f"  Columns po standaryzacji: {list(df.columns)}")

    df['invoice']      = df['invoice'].astype(str).str.strip()
    df['stock_code']   = df['stock_code'].astype(str).str.strip().str.upper()
    df['description']  = (df['description']
                          .fillna('Unknown')
                          .astype(str).str.strip().str.title())
    df['country']      = df['country'].astype(str).str.strip()
    df['invoice_date'] = pd.to_datetime(df['invoice_date'])

    df['is_registered'] = df['customer_id'].notna()

    n_registered = df['is_registered'].sum()
    n_guests     = (~df['is_registered']).sum()
    guest_pct    = n_guests / len(df) * 100

    df['customer_id'] = df['customer_id'].fillna(0).astype(int)

    log.info(f"  Registered: {n_registered:>9,}  ({100 - guest_pct:.1f}%)")
    log.info(f"  Guests → id=0:  {n_guests:>9,}  ({guest_pct:.1f}%)")
    audit.append(('po guest flagging (0 loss)', len(df)))

    df['is_return'] = df['invoice'].str.startswith('C')
    log.info(f"  Returns flagged: {df['is_return'].sum():>9,}")

    mask_non_product = df['stock_code'].isin(EXCLUDE_CODES)
    df = df[~mask_non_product].copy()
    audit.append(('after drop non-product codes', len(df)))


    bad_price = (df['price'] < 0) & (~df['is_return'])
    df = df[~bad_price].copy()
    audit.append(('after drop price < 0', len(df)))

    zero_qty = df['quantity'] == 0
    df = df[~zero_qty].copy()
    audit.append(('after drop qty == 0', len(df)))


    df.drop_duplicates(inplace=True)
    audit.append(('after deduplication', len(df)))


    df['quantity_abs']   = df['quantity'].abs()
    df['line_total']     = (df['quantity'] * df['price']).round(2)
    df['line_total_abs'] = df['line_total'].abs().round(2)
    df['date_key']       = df['invoice_date'].dt.strftime('%Y%m%d').astype(int)
    df['hour']           = df['invoice_date'].dt.hour

    log.info("")
    log.info("  ┌──────────────────────────────────┬──────────┬─────────┐")
    log.info("  │ Step                             │   Rows│   Loss│")
    log.info("  ├──────────────────────────────────┼──────────┼─────────┤")
    for i, (step, n) in enumerate(audit):
        if i == 0:
            loss_str = "      —"
        else:
            loss = audit[i - 1][1] - n
            loss_str = f"-{loss:>6,}" if loss > 0 else "      0"
        log.info(f"  │ {step:32s} │ {n:>8,} │ {loss_str} │")
    log.info("  └──────────────────────────────────┴──────────┴─────────┘")

    raw_n   = audit[0][1]
    final_n = audit[-1][1]
    pct     = final_n / raw_n * 100
    log.info(f"  Retention:  {final_n:,} / {raw_n:,} = {pct:.1f}%")
    log.info(f"  Returns:    {df['is_return'].sum():,}")
    log.info(f"  Guests:    {(~df['is_registered']).sum():,} lines "
             f"({(~df['is_registered']).mean() * 100:.1f}%)")
    log.info(f"  Period:     {df['invoice_date'].min().date()} → "
             f"{df['invoice_date'].max().date()}")
    log.info("")

    return df

---

## 5. DIMENSIONS — Budowa tabel wymiarów <a id="5-dimensions"></a>
### 📅 5a. `dim_date` — Wymiar daty


In [26]:
def build_dim_date(df: pd.DataFrame) -> pd.DataFrame:
    """Date dimension — full calendar."""
    log.info("═══ BUILD dim_date ════════════════════════════════")

    d_min = df['invoice_date'].min().normalize()
    d_max = df['invoice_date'].max().normalize()

    dates = pd.date_range(d_min, d_max, freq='D')
    dim = pd.DataFrame({'full_date': dates})

    dim['date_key']     = dim['full_date'].dt.strftime('%Y%m%d').astype(int)
    dim['year']         = dim['full_date'].dt.year
    dim['quarter']      = dim['full_date'].dt.quarter
    dim['month']        = dim['full_date'].dt.month
    dim['day']          = dim['full_date'].dt.day
    dim['day_of_week']  = dim['full_date'].dt.dayofweek + 1
    dim['day_name']     = dim['full_date'].dt.day_name()
    dim['month_name']   = dim['full_date'].dt.month_name()
    dim['year_month']   = dim['full_date'].dt.to_period('M').astype(str)
    dim['year_quarter'] = dim['full_date'].dt.to_period('Q').astype(str)
    dim['is_month_end'] = dim['full_date'].dt.is_month_end

    dim['full_date'] = dim['full_date'].dt.date
    
    dim = dim[
        [
            "date_key",
            "full_date",
            "year",
            "quarter",
            "month",
            "month_name",
            "day",
            "day_of_week",
            "day_name",
            "year_month",
            "year_quarter",
            "is_month_end",
        ]
    ]

    log.info(f"  {len(dim):,} days  ({d_min.date()} → {d_max.date()})")
    return dim

### 📦 5c. `dim_product` — Wymiar produktu


In [27]:
def build_dim_product(df: pd.DataFrame) -> pd.DataFrame:
    """Product dimension — description + sales price statistics."""
    log.info("═══ BUILD dim_product ═════════════════════════════")

    sales = df[~df["is_return"]].copy()
    
    product = (
        df.groupby("stock_code", as_index=False)
        .agg(
            product_name=("description", lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]),
        )
        .sort_values("stock_code")
        .reset_index(drop=True)
    )
    
    price_stats = (
        sales.groupby("stock_code", as_index=False)
        .agg(unit_price_median=("price", "median"))
    )
    
    dim = product.merge(price_stats, on="stock_code", how="left")
    dim["unit_price_median"] = dim["unit_price_median"].fillna(0).round(2)

    dim.insert(0, "product_key", range(1, len(dim) + 1))

    log.info(f"  {len(dim):,} products")
    return dim

### 👤 5d. `dim_customer` — Customer dimension



In [ ]:
def r_score_from_days(days_since: pd.Series) -> pd.Series:
    return np.select(
        [days_since <= 30, days_since <= 90],
        [3, 2],
        default=1,
    )

def f_score_from_orders(total_orders: pd.Series) -> pd.Series:
    return np.select(
        [total_orders >= 20, total_orders >= 5],
        [3, 2],
        default=1,
    )

def segment_name_from_scores(r_score: pd.Series, f_score: pd.Series) -> pd.Series:
    conditions = [
        (r_score == 3) & (f_score == 3),
        (r_score >= 2) & (f_score >= 2),
        (r_score == 3) & (f_score == 1),
        (r_score == 2) & (f_score == 1),
        (r_score == 1) & (f_score >= 2),
        (r_score == 1) & (f_score == 1),
    ]
    values = [
        "Champions",
        "Loyal",
        "Recent Buyers",
        "Promising",
        "At Risk",
        "Lost",
    ]
    return pd.Series(np.select(conditions, values, default="Promising"), index=r_score.index)

def compute_risk_score(
    segment_name: pd.Series,
    days_since_last_purchase: pd.Series,
    total_transactions: pd.Series,
) -> pd.Series:
    base_map = {
        "Champions": 10,
        "Loyal": 20,
        "Recent Buyers": 35,
        "Promising": 50,
        "At Risk": 75,
        "Lost": 90,
        "Guest": 0,
    }
    base = segment_name.map(base_map).fillna(0)
    recency_adjustment = (days_since_last_purchase.clip(lower=0, upper=180) / 180 * 10).round()
    frequency_adjustment = np.select(
        [total_transactions >= 20, total_transactions >= 5],
        [-5, 0],
        default=5,
    )
    return (base + recency_adjustment + frequency_adjustment).clip(lower=0, upper=100).astype(int)

def risk_tier_from_score(risk_score: pd.Series) -> pd.Series:
    return pd.Series(
        np.select(
            [risk_score < 25, risk_score < 50, risk_score < 75],
            ["Healthy", "Watchlist", "High Risk"],
            default="Critical",
        ),
        index=risk_score.index,
    )

def recommended_action(segment_name: pd.Series, value_tier: pd.Series) -> pd.Series:
    conditions = [
        segment_name == "Champions",
        segment_name == "Loyal",
        segment_name == "Recent Buyers",
        segment_name == "Promising",
        (segment_name == "At Risk") & value_tier.isin(["VIP", "High Value"]),
        segment_name == "At Risk",
        segment_name == "Lost",
    ]
    values = [
        "Protect loyalty",
        "Upsell / cross-sell",
        "Second-order nudge",
        "Nurture journey",
        "Personal outreach",
        "Retention offer",
        "Win-back campaign",
    ]
    return pd.Series(np.select(conditions, values, default="Monitor"), index=segment_name.index)

def build_dim_customer(df: pd.DataFrame, dim_segment: pd.DataFrame) -> pd.DataFrame:
    """Customer dimension (PRO) - new model with risk score and predictions."""
    log.info("═══ BUILD dim_customer ════════════════════════════")

    sales = df[df["is_registered"] & ~df["is_return"]].copy()
    customer = (
        sales.groupby("customer_id", as_index=False)
        .agg(
            first_purchase_date=("invoice_date", "min"),
            last_purchase_date=("invoice_date", "max"),
            total_transactions=("invoice", "nunique"),
            total_items=("quantity_abs", "sum"),
            total_revenue=("line_total", "sum"),
        )
        .sort_values("customer_id")
        .reset_index(drop=True)
    )

    reference_date = df["invoice_date"].max().normalize() + pd.Timedelta(days=1)
    customer["acquisition_month"] = customer["first_purchase_date"].dt.to_period("M").astype(str)
    customer["is_registered"] = True
    customer["avg_order_value"] = (customer["total_revenue"] / customer["total_transactions"]).round(2)
    customer["days_since_last_purchase"] = (reference_date - customer["last_purchase_date"].dt.normalize()).dt.days.astype(int)
    customer["active_90d_flag"] = customer["days_since_last_purchase"] <= 90

    vip_threshold = customer["total_revenue"].quantile(0.90)
    high_threshold = customer["total_revenue"].quantile(0.75)
    customer["value_tier"] = "Standard"
    customer.loc[customer["total_revenue"] >= high_threshold, "value_tier"] = "High Value"
    customer.loc[customer["total_revenue"] >= vip_threshold, "value_tier"] = "VIP"

    customer["rfm_r_score"] = r_score_from_days(customer["days_since_last_purchase"])
    customer["rfm_f_score"] = f_score_from_orders(customer["total_transactions"])
    customer["segment_name"] = segment_name_from_scores(customer["rfm_r_score"], customer["rfm_f_score"])

    segment_lookup = dim_segment.set_index("segment_name")["segment_key"]
    customer["segment_key"] = customer["segment_name"].map(segment_lookup).astype(int)
    
    customer["clv_proxy"] = customer["total_revenue"].round(2)
    customer["risk_score"] = compute_risk_score(
        customer["segment_name"],
        customer["days_since_last_purchase"],
        customer["total_transactions"],
    )
    customer["risk_tier"] = risk_tier_from_score(customer["risk_score"])
    customer["recommended_action"] = recommended_action(customer["segment_name"], customer["value_tier"])

    customer.insert(0, "customer_key", range(1, len(customer) + 1))
    customer["total_revenue"] = customer["total_revenue"].round(2)

    guest_row = pd.DataFrame(
        [
            {
                "customer_key": 0,
                "customer_id": 0,
                "is_registered": False,
                "first_purchase_date": pd.NaT,
                "last_purchase_date": pd.NaT,
                "acquisition_month": "Unknown",
                "total_transactions": 0,
                "total_items": 0,
                "total_revenue": 0.0,
                "avg_order_value": 0.0,
                "days_since_last_purchase": 0,
                "active_90d_flag": False,
                "value_tier": "Standard",
                "rfm_r_score": 0,
                "rfm_f_score": 0,
                "segment_key": 0, 
                "clv_proxy": 0.0,
                "risk_score": 0,
                "risk_tier": "Healthy",
                "recommended_action": "Monitor",
            }
        ]
    )

    customer = pd.concat([guest_row, customer], ignore_index=True)
    
    col_order = [
        "customer_key", "customer_id", "is_registered", "first_purchase_date",
        "last_purchase_date", "acquisition_month", "total_transactions",
        "total_items", "total_revenue", "avg_order_value",
        "days_since_last_purchase", "active_90d_flag", "value_tier",
        "rfm_r_score", "rfm_f_score", "segment_key", "clv_proxy",
        "risk_score", "risk_tier", "recommended_action",
    ]
    
    dim = customer[col_order]
    
    log.info(f"  {len(dim):,} customers (including 1 guest row)")
    return dim

In [ ]:
SEGMENT_DEFINITION = [
    {
        "segment_key": 1,
        "segment_name": "Champions",
        "segment_group": "Core",
        "sort_order": 1,
        "lifecycle_rank": 6,
    },
    {
        "segment_key": 2,
        "segment_name": "Loyal",
        "segment_group": "Core",
        "sort_order": 2,
        "lifecycle_rank": 5,
    },
    {
        "segment_key": 3,
        "segment_name": "Recent Buyers",
        "segment_group": "Growth",
        "sort_order": 3,
        "lifecycle_rank": 4,
    },
    {
        "segment_key": 4,
        "segment_name": "Promising",
        "segment_group": "Growth",
        "sort_order": 4,
        "lifecycle_rank": 3,
    },
    {
        "segment_key": 5,
        "segment_name": "At Risk",
        "segment_group": "Risk",
        "sort_order": 5,
        "lifecycle_rank": 2,
    },
    {
        "segment_key": 6,
        "segment_name": "Lost",
        "segment_group": "Risk",
        "sort_order": 6,
        "lifecycle_rank": 1,
    },
    {
        "segment_key": 0,
        "segment_name": "Guest",
        "segment_group": "Other",
        "sort_order": 7,
        "lifecycle_rank": 0,
    },
]

STALE_TABLES = {
    "dim_country",
    "analytics_customer_pareto",
    "analytics_customer_migration",
}

def build_dim_segment() -> pd.DataFrame:
    return pd.DataFrame(SEGMENT_DEFINITION)

---

## 6. FACTS — Fact table building <a id="6-facts"></a>

### 🧾 6a. `fct_order_lines` — Linie orders

Najniższy poziom granularności — **1 wiersz = 1 pozycja na fakturze**.

In [ ]:
def build_fct_order_lines(
    df: pd.DataFrame,
    dim_customer: pd.DataFrame,
    dim_product: pd.DataFrame
) -> pd.DataFrame:
    """
    Fact table — grain: 1 order line.
    Goście → customer_key = 0 (sentinel).
    """
    log.info("═══ BUILD fct_order_lines ═════════════════════════")

    fct = df.copy()

    customer_lookup = dim_customer.set_index("customer_id")["customer_key"]
    fct["customer_key"] = fct["customer_id"].map(customer_lookup).fillna(0).astype(int)

    product_lookup = dim_product.set_index("stock_code")["product_key"]
    fct["product_key"] = fct["stock_code"].map(product_lookup).astype(int)

    fct = fct.reset_index(drop=True)

    fct.insert(0, "order_line_key", range(1, len(fct) + 1))

    fct = fct[
        [
            "order_line_key",
            "invoice",
            "customer_key",
            "product_key",
            "date_key",
            "hour",
            "quantity",
            "quantity_abs",
            "price",
            "line_total",
            "line_total_abs",
            "is_return",
        ]
    ].copy()

    n_reg   = (fct["customer_key"] > 0).sum()
    n_guest = (fct["customer_key"] == 0).sum()
    log.info(f"  {len(fct):,} order lines")
    log.info(f"    Registered (key>0): {n_reg:>9,}  ({n_reg / len(fct) * 100:.1f}%)")
    log.info(f"    Guest      (key=0): {n_guest:>9,}  ({n_guest / len(fct) * 100:.1f}%)")
    log.info(f"    Sales:           {(~fct['is_return']).sum():>9,}")
    log.info(f"    Returns:             {fct['is_return'].sum():>9,}")

    return fct

### 📋 6b. `fct_orders` — Aggregated orders

Zagregowana tabela faktów — **1 wiersz = 1 faktura**.

In [ ]:
def build_fct_orders(fct_lines: pd.DataFrame) -> pd.DataFrame:
    """Fact table — grain: 1 invoice (aggregated lines)."""
    log.info("═══ BUILD fct_orders ══════════════════════════════")

    orders = (
        fct_lines.groupby("invoice", as_index=False)
        .agg(
            customer_key=("customer_key", "first"),
            date_key=("date_key", "first"),
            hour=("hour", "first"),
            total_quantity=("quantity", "sum"),
            total_items=("quantity_abs", "sum"),
            total_revenue=("line_total", "sum"),
            gross_revenue=("line_total_abs", "sum"),
            num_lines=("order_line_key", "count"),
            num_unique_products=("product_key", "nunique"),
            is_return=("is_return", "first"),
        )
        .sort_values(["date_key", "invoice", "hour"])
        .reset_index(drop=True)
    )

    orders["total_revenue"] = orders["total_revenue"].round(2)
    orders["gross_revenue"] = orders["gross_revenue"].round(2)
    orders.insert(0, "order_key", range(1, len(orders) + 1))

    sales = orders[~orders["is_return"]]
    n_reg = (orders["customer_key"] > 0).sum()
    n_guest = (orders["customer_key"] == 0).sum()
    
    log.info(f"  {len(orders):,} orders")
    log.info(f"    Sales:     {(~orders['is_return']).sum():>8,}")
    log.info(f"    Returns:       {orders['is_return'].sum():>8,}")
    log.info(f"    Registered:   {n_reg:>8,}")
    log.info(f"    Guest:        {n_guest:>8,}")
    log.info(f"    Revenue:   £{sales['total_revenue'].sum():>12,.2f}")

    return orders[
        [
            "order_key",
            "invoice",
            "customer_key",
            "date_key",
            "hour",
            "total_quantity",
            "total_items",
            "total_revenue",
            "gross_revenue",
            "num_lines",
            "num_unique_products",
            "is_return",
        ]
    ]

In [32]:
def attach_order_keys(
    fct_order_lines: pd.DataFrame, fct_orders: pd.DataFrame
) -> pd.DataFrame:
    order_keys = fct_orders[["order_key", "invoice"]]
    lines = fct_order_lines.merge(order_keys, on="invoice", how="left")
    return lines[
        [
            "order_line_key",
            "order_key",
            "customer_key",
            "product_key",
            "date_key",
            "quantity",
            "quantity_abs",
            "price",
            "line_total",
            "line_total_abs",
            "is_return",
        ]
    ]

In [33]:
def build_analytics_customer_pareto(dim_customer: pd.DataFrame) -> pd.DataFrame:
    log.info("=== BUILD analytics_customer_pareto ===================")
    reg = dim_customer[dim_customer['is_registered'] & (dim_customer['total_revenue'] > 0)].copy()
    reg = reg.sort_values('total_revenue', ascending=False).reset_index(drop=True)
    total_revenue = reg['total_revenue'].sum()
    n_customers = len(reg)
    pareto = pd.DataFrame({
        'customer_key': reg['customer_key'].values,
        'customer_id': reg['customer_id'].values,
        'rfm_segment': reg['rfm_segment'].values,
        'value_tier': reg['value_tier'].values,
        'total_revenue': reg['total_revenue'].values,
        'customer_rank': reg['total_revenue'].rank(method='dense', ascending=False).astype(int)
    })
    pareto = pareto.sort_values('customer_rank').reset_index(drop=True)
    pareto['cumulative_revenue'] = pareto['total_revenue'].cumsum().round(2)
    pareto['cumulative_revenue_pct'] = (pareto['cumulative_revenue'] / total_revenue * 100).round(4)
    pareto['cumulative_customer_pct'] = ((np.arange(1, n_customers + 1) / n_customers) * 100).round(4)
    return pareto


### 📋 6c. `fct_customer_migration` — historical segment changes

In [ ]:
def build_fct_customer_migration(
    df: pd.DataFrame, dim_customer: pd.DataFrame, dim_segment: pd.DataFrame
) -> pd.DataFrame:
    log.info("=== BUILD fct_customer_migration (MONTHLY) ========")
    sales = df[df["is_registered"] & ~df["is_return"]].copy()
    sales["year_month"] = sales["invoice_date"].dt.to_period("M")
    months = sorted(sales["year_month"].unique())

    if len(months) < 2:
        return pd.DataFrame(
            columns=[
                "migration_key",
                "customer_key",
                "from_date_key",
                "to_date_key",
                "segment_from_key",
                "segment_to_key",
                "migration_direction",
                "revenue_from",
                "revenue_to",
            ]
        )

    segment_lookup = dim_segment.set_index("segment_name")
    customer_lookup = dim_customer.set_index("customer_id")["customer_key"]
    monthly_snapshots = []

    for month_period in months:
        month_end = month_period.to_timestamp(how="end").normalize()
        history = sales[sales["invoice_date"] <= month_end]

        snapshot = (
            history.groupby("customer_id", as_index=False)
            .agg(
                last_purchase=("invoice_date", "max"),
                total_transactions=("invoice", "nunique"),
            )
            .sort_values("customer_id")
        )
        days_since = (month_end - snapshot["last_purchase"].dt.normalize()).dt.days
        
        snapshot["rfm_r_score"] = r_score_from_days(days_since)
        snapshot["rfm_f_score"] = f_score_from_orders(snapshot["total_transactions"])
        snapshot["segment_name"] = segment_name_from_scores(
            snapshot["rfm_r_score"], snapshot["rfm_f_score"]
        )
        snapshot["segment_key"] = (
            snapshot["segment_name"]
            .map(segment_lookup["segment_key"])
            .astype(int)
        )

        month_sales = (
            sales[sales["year_month"] == month_period]
            .groupby("customer_id", as_index=False)["line_total"]
            .sum()
            .rename(columns={"line_total": "month_revenue"})
        )
        snapshot = snapshot.merge(month_sales, on="customer_id", how="left")
        snapshot["month_revenue"] = snapshot["month_revenue"].fillna(0).round(2)
        snapshot["date_key"] = int(month_end.strftime("%Y%m%d"))
        snapshot["year_month"] = str(month_period)

        monthly_snapshots.append(
            snapshot[
                [
                    "customer_id",
                    "date_key",
                    "year_month",
                    "segment_key",
                    "month_revenue",
                ]
            ]
        )

    snapshots = pd.concat(monthly_snapshots, ignore_index=True)
    lifecycle_rank = segment_lookup["lifecycle_rank"]
    migrations = []

    for current_idx in range(1, len(months)):
        previous_month = str(months[current_idx - 1])
        current_month = str(months[current_idx])

        previous_snapshot = snapshots[snapshots["year_month"] == previous_month].rename(
            columns={
                "date_key": "from_date_key",
                "segment_key": "segment_from_key",
                "month_revenue": "revenue_from",
            }
        )
        current_snapshot = snapshots[snapshots["year_month"] == current_month].rename(
            columns={
                "date_key": "to_date_key",
                "segment_key": "segment_to_key",
                "month_revenue": "revenue_to",
            }
        )

        pair = previous_snapshot.merge(
            current_snapshot[
                ["customer_id", "to_date_key", "segment_to_key", "revenue_to"]
            ],
            on="customer_id",
            how="inner",
        )
        
        pair["from_rank"] = pair["segment_from_key"].map(lifecycle_rank).fillna(0)
        pair["to_rank"] = pair["segment_to_key"].map(lifecycle_rank).fillna(0)
        
        pair["migration_direction"] = np.select(
            [pair["to_rank"] > pair["from_rank"], pair["to_rank"] < pair["from_rank"]],
            ["Upgrade", "Downgrade"],
            default="Stable",
        )
        
        pair["customer_key"] = pair["customer_id"].map(customer_lookup).astype(int)
        migrations.append(
            pair[
                [
                    "customer_key",
                    "from_date_key",
                    "to_date_key",
                    "segment_from_key",
                    "segment_to_key",
                    "migration_direction",
                    "revenue_from",
                    "revenue_to",
                ]
            ]
        )

    migration = pd.concat(migrations, ignore_index=True)
    migration.insert(0, "migration_key", range(1, len(migration) + 1))
    
    log.info(f"  Generated {len(migration):,} rows migracji.")
    return migration

---

## 7. SAVE — Saving results <a id="7-save"></a>

In [ ]:
def save_table(name: str, table: pd.DataFrame) -> None:
    csv_path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(csv_path, index=False, encoding="utf-8-sig")
    log.info("Saved %s.csv (%s rows).", name, f"{len(table):,}")

def purge_stale_outputs(active_tables: Dict[str, pd.DataFrame]) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    stale_names = STALE_TABLES.union(
        {
            file_path.stem
            for file_path in OUTPUT_DIR.glob("*.csv")
            if file_path.stem not in active_tables
        }
    )
    for table_name in sorted(stale_names):
        path = OUTPUT_DIR / f"{table_name}.csv"
        if path.exists():
            path.unlink()
            log.info("Deleted stale artifact %s.", path.name)

---

## 8. QUALITY CHECK — Quality check <a id="8-quality-check"></a>


In [36]:
def quality_summary(
    dim_customer: pd.DataFrame, dim_segment: pd.DataFrame, fct_orders: pd.DataFrame
) -> None:
    segment_names = dim_segment.set_index("segment_key")["segment_name"]
    customer_summary = (
        dim_customer[dim_customer["is_registered"]]
        .assign(segment_name=lambda x: x["segment_key"].map(segment_names))
        .groupby("segment_name", as_index=False)
        .agg(
            customers=("customer_key", "count"),
            revenue=("total_revenue", "sum"),
        )
        .sort_values("customers", ascending=False)
    )
    log.info("Final segmentation:")
    for row in customer_summary.itertuples(index=False):
        log.info(
            "  %-14s | customers: %5s | revenue: %10s",
            row.segment_name,
            f"{row.customers:,}",
            f"GBP {row.revenue:,.0f}",
        )
    log.info(
        "Orders: %s | Return rate: %.2f%%",
        f"{len(fct_orders):,}",
        fct_orders["is_return"].mean() * 100,
    )

---

## 9. Pipeline execution <a id="9-execution"></a>

In [37]:
def main() -> Dict[str, pd.DataFrame]:
    log.info("Starting build of simplified model pod 3 dashboardy.")
    raw = extract()
    clean_df = clean(raw)

    dim_segment = build_dim_segment()
    dim_date = build_dim_date(clean_df)
    dim_product = build_dim_product(clean_df)
    dim_customer = build_dim_customer(clean_df, dim_segment)
    
    fct_order_lines = build_fct_order_lines(clean_df, dim_customer, dim_product)
    fct_orders = build_fct_orders(fct_order_lines)
    fct_order_lines = attach_order_keys(fct_order_lines, fct_orders)
    
    fct_customer_migration = build_fct_customer_migration(
        clean_df, dim_customer, dim_segment
    )

    tables = {
        "dim_date": dim_date,
        "dim_segment": dim_segment,
        "dim_product": dim_product,
        "dim_customer": dim_customer,
        "fct_orders": fct_orders,
        "fct_order_lines": fct_order_lines,
        "fct_customer_migration": fct_customer_migration,
    }

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    purge_stale_outputs(tables)
    
    for name, table in tables.items():
        save_table(name, table)

    quality_summary(dim_customer, dim_segment, fct_orders)
    log.info("Model ready. Output folder: %s", OUTPUT_DIR)
    
    return tables

### ▶️ Run pipeline

In [38]:
tables = main()

13:57:24 │ INFO    │ Start budowy uproszczonego modelu pod 3 dashboardy.
13:57:24 │ INFO    │ ═══ EXTRACT ═══════════════════════════════════════
13:57:52 │ INFO    │   Arkusz 'Year 2009-2010':   525,461 wierszy
13:58:20 │ INFO    │   Arkusz 'Year 2010-2011':   541,910 wierszy
13:58:20 │ INFO    │   RAZEM:                   1,067,371 wierszy
13:58:20 │ INFO    │   Kolumny: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
13:58:20 │ INFO    │ ═══ CLEAN ═════════════════════════════════════════
13:58:21 │ INFO    │   Kolumny po standaryzacji: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country']
13:58:22 │ INFO    │   Zarejestrowani:   824,364  (77.2%)
13:58:22 │ INFO    │   Goście → id=0:    243,007  (22.8%)
13:58:22 │ INFO    │   Zwroty oflagowane:    19,494
13:58:29 │ INFO    │ 
13:58:29 │ INFO    │   ┌──────────────────────────────────┬──────────┬─────────┐
13:58:29 │ INFO    │   │ 